# ViT-aligned ResNet-50 benchmark experiments

This notebook applies the corrected ViT benchmark protocol to a fully fine-tuned ImageNet-pretrained ResNet-50. It uses the exact same 96 experiment manifests, eight held-out generators, k=1/2/4/7, three dataset repeats, seed 17, and independent baseline/augmented treatments (192 runs total). ResNet retains ImageNet normalization because its pretrained weights expect different statistics than CLIP.

- `benchmark/experiments.json` supplies the frozen experiment metadata.
- `benchmark/manifests/<experiment>/train.csv` updates model weights.
- `benchmark/manifests/<experiment>/val.csv` selects the best checkpoint.
- `benchmark/manifests/<experiment>/test.csv` supplies final held-out-generator metrics.
- Google Drive paths are remapped to the cluster's local `GenImage_Compact` root.


In [ ]:
# Run only if packages are missing.
# %pip install -q torch torchvision pandas numpy scikit-learn matplotlib seaborn pillow tqdm

In [ ]:
from __future__ import annotations

import gc
import io
import hashlib
import json
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image, ImageFile, ImageFilter
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    raise RuntimeError("Select a GPU-backed Jupyter kernel before running the CNN benchmark.")
x = torch.ones((16, 16), device=DEVICE)
assert (x @ x).sum().item() == 4096
torch.cuda.synchronize()
print("GPU operation passed:", torch.cuda.get_device_name(0))

## 1. Configuration

In [ ]:
@dataclass
class Config:
    data_root: str = "/project2/yzhao010_1531/csci566-genimage/data/extracted/GenImage_Compact"
    benchmark_root: str = "/project2/yzhao010_1531/csci566-genimage/data/extracted/GenImage_Compact/benchmark"
    experiment_root: str = "/project2/yzhao010_1531/csci566-genimage/data/extracted/GenImage_Compact/benchmark/manifests"
    output_root: str = "/project2/yzhao010_1531/csci566-genimage/results/cnn_benchmark/notebook"

    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 0
    epochs: int = 20
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    early_stopping_patience: int = 3
    pretrained: bool = True
    use_amp: bool = True
    decision_threshold: float = 0.5
    base_seed: int = 17
    auroc_tolerance: float = 1e-12
    calibration_bins: int = 10
    verify_image_hashes: bool = True
    skip_completed: bool = True
    max_runs_this_session: Optional[int] = None

    # Optional fast integration test. It temporarily subsamples the supplied CSVs.
    # Keep None for real experiments so the exact dataset-team splits are used.
    debug_max_rows_per_split: Optional[int] = None


CFG = Config()

# One or more metadata files; each must contain a JSON list of experiments.
EXPERIMENT_PLAN_FILES: list[str] = [
    f"{CFG.benchmark_root}/experiments.json",
]

# ["ALL"] runs everything. Otherwise list any one or several exact names.
SELECTED_EXPERIMENTS: list[str] = [
    "ALL",
]

# Choose either variant separately or both. Order controls execution order.
TRAINING_VARIANTS: list[str] = [
    "baseline",
    "augmented",
]

print(asdict(CFG))
print("Plan files:", EXPERIMENT_PLAN_FILES)
print("Requested experiments:", SELECTED_EXPERIMENTS)

Every corrected manifest is required to contain 1,400 real + 1,400 fake training images and 280 real + 280 fake validation/test images. No resampling is performed.

## 2. Reproducibility and CSV path remapping

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def relative_genimage_path(raw_path: str) -> str:
    text = str(raw_path).replace("\\", "/")
    marker = re.search(r"(?:^|/)GenImage_Compact(?:_shared)?/", text)
    if marker:
        return text[marker.end():]

    path = Path(text)
    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.relative_to(Path(CFG.data_root)).as_posix()
    except ValueError as exc:
        raise ValueError(f"Cannot map CSV path to GenImage_Compact: {raw_path}") from exc


def debug_cap(frame: pd.DataFrame, maximum: Optional[int], seed: int) -> pd.DataFrame:
    if maximum is None or len(frame) <= maximum:
        return frame.copy()
    # Debug-only stratification; full runs always preserve the CSV exactly.
    parts = []
    labels = sorted(frame.label.unique())
    base, remainder = divmod(maximum, len(labels))
    for i, label in enumerate(labels):
        count = base + int(i < remainder)
        pool = frame[frame.label == label]
        parts.append(pool.sample(n=min(count, len(pool)), random_state=seed + i))
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed)


VERIFIED_CONTENT: dict[tuple[str, int, int], str] = {}


def file_digest(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_split_csv(csv_path: Path, split_name: str, seed: int) -> pd.DataFrame:
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing experiment split: {csv_path}")
    frame = pd.read_csv(csv_path)
    required = {"path", "label", "source_generator", "sha256"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")

    frame = frame.copy()
    frame["original_path"] = frame["path"].astype(str)
    frame["path"] = frame["original_path"].map(relative_genimage_path)
    frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(int)
    if not set(frame.label.unique()).issubset({0, 1}):
        raise ValueError(f"{csv_path}: labels must be 0 or 1.")
    frame["label_name"] = frame.label.map({0: "nature", 1: "ai"})
    frame["generator"] = frame["source_generator"].astype(str)
    frame["official_split"] = (
        frame["source_split"].astype(str).str.lower()
        if "source_split" in frame else "not_recorded"
    )
    frame["experiment_split"] = split_name

    frame["sha256"] = frame.sha256.astype(str).str.lower()
    if (frame.sha256.isna().any() or frame.sha256.duplicated().any()
            or not frame.sha256.str.fullmatch(r"[0-9a-f]{64}").all()):
        raise ValueError(f"{csv_path}: missing or duplicate sha256 values within this split.")

    missing_files = []
    for row in frame.itertuples():
        path = Path(CFG.data_root) / row.path
        if not path.is_file():
            missing_files.append(str(path))
            continue
        if CFG.verify_image_hashes:
            stat = path.stat()
            key = (str(path), stat.st_size, stat.st_mtime_ns)
            if key not in VERIFIED_CONTENT:
                VERIFIED_CONTENT[key] = file_digest(path)
            if VERIFIED_CONTENT[key] != row.sha256:
                raise ValueError(f"Content differs from manifest: {path}")
    if missing_files:
        preview = missing_files[:5]
        raise FileNotFoundError(
            f"{csv_path}: {len(missing_files)} remapped image files do not exist. "
            f"First examples: {preview}"
        )

    return debug_cap(frame, CFG.debug_max_rows_per_split, seed)

## 3. Load and select experiment specifications

In [ ]:
def load_experiment_plans(plan_files: Sequence[str]) -> list[dict]:
    if isinstance(plan_files, str):
        raise TypeError("EXPERIMENT_PLAN_FILES must be a list of strings, not one string.")
    experiments = []
    for raw_path in plan_files:
        if not isinstance(raw_path, str):
            raise TypeError("Every plan-file entry must be a string.")
        path = Path(raw_path)
        with path.open() as file:
            entries = json.load(file)
        if not isinstance(entries, list):
            raise ValueError(f"{path} must contain a JSON list.")
        for entry in entries:
            if not isinstance(entry, dict):
                raise ValueError(f"Every entry in {path} must be a JSON object.")
            entry = dict(entry)
            entry["_plan_file"] = str(path)
            experiments.append(entry)

    names = [entry.get("experiment") for entry in experiments]
    duplicates = pd.Series(names).duplicated(keep=False)
    if duplicates.any():
        repeated = sorted(set(pd.Series(names)[duplicates]))
        raise ValueError(f"Duplicate experiment names across plans: {repeated}")
    return experiments


def validate_experiment(entry: dict, available: Sequence[str]) -> None:
    required = {"experiment", "holdout", "k", "rep", "train_generators"}
    missing = required - set(entry)
    if missing:
        raise ValueError(f"{entry.get('experiment', '<unnamed>')} is missing {sorted(missing)}")
    if not isinstance(entry["train_generators"], list) or not all(
        isinstance(x, str) for x in entry["train_generators"]
    ):
        raise TypeError("train_generators must be a list of strings.")
    if len(entry["train_generators"]) != int(entry["k"]):
        raise ValueError(f"{entry['experiment']}: k does not match train_generators length.")
    if len(set(entry["train_generators"])) != len(entry["train_generators"]):
        raise ValueError(f"{entry['experiment']}: duplicate training generators.")
    if entry["holdout"] in entry["train_generators"]:
        raise ValueError(f"{entry['experiment']}: holdout appears in training generators.")
    referenced = set(entry["train_generators"]) | {entry["holdout"]}
    unknown = referenced - set(available)
    if unknown:
        raise ValueError(f"{entry['experiment']}: unknown generators {sorted(unknown)}")
    if int(entry["k"]) not in {1, 2, 4, 7} or int(entry["rep"]) not in {0, 1, 2}:
        raise ValueError(f"{entry['experiment']}: expected k in 1/2/4/7 and rep in 0/1/2.")
    expected_name = f"holdout_{entry['holdout']}_k{entry['k']}_rep{entry['rep']}"
    if entry["experiment"] != expected_name:
        raise ValueError(f"Experiment identity mismatch: {entry['experiment']} != {expected_name}")


EXPECTED_GENERATORS = {
    "ADM", "BigGAN", "glide", "Midjourney", "stable_diffusion_v_1_4",
    "stable_diffusion_v_1_5", "VQDM", "wukong",
}
available_generators = sorted(
    p.name for p in Path(CFG.data_root).iterdir()
    if p.is_dir() and p.name in EXPECTED_GENERATORS
)
if set(available_generators) != EXPECTED_GENERATORS:
    raise ValueError(f"Generator folders mismatch: missing {sorted(EXPECTED_GENERATORS - set(available_generators))}")
all_experiments = load_experiment_plans(EXPERIMENT_PLAN_FILES)
for experiment in all_experiments:
    validate_experiment(experiment, available_generators)

if len(all_experiments) != 96:
    raise ValueError(f"Expected 96 corrected benchmark experiments, found {len(all_experiments)}.")
expected_grid = {(g, k, r) for g in EXPECTED_GENERATORS for k in (1, 2, 4, 7) for r in (0, 1, 2)}
actual_grid = {(entry["holdout"], int(entry["k"]), int(entry["rep"])) for entry in all_experiments}
if actual_grid != expected_grid:
    raise ValueError("Corrected benchmark does not contain the complete holdout/k/repeat grid.")

if not isinstance(SELECTED_EXPERIMENTS, list) or not all(
    isinstance(name, str) for name in SELECTED_EXPERIMENTS
):
    raise TypeError("SELECTED_EXPERIMENTS must be a list of strings.")

if len(SELECTED_EXPERIMENTS) == 1 and SELECTED_EXPERIMENTS[0].upper() == "ALL":
    selected_experiments = all_experiments
else:
    requested = set(SELECTED_EXPERIMENTS)
    known = {entry["experiment"] for entry in all_experiments}
    unknown = requested - known
    if unknown:
        raise ValueError(f"Unknown requested experiment names: {sorted(unknown)}")
    selected_experiments = [
        entry for entry in all_experiments if entry["experiment"] in requested
    ]

if not selected_experiments:
    raise ValueError("No experiments selected.")

print(f"Selected {len(selected_experiments)} of {len(all_experiments)} experiments:")
for entry in selected_experiments:
    print(" -", entry["experiment"])

## 4. Load and validate the exact experiment CSV splits

In [ ]:
def load_experiment_splits(entry: dict):
    seed = CFG.base_seed
    experiment_dir = Path(CFG.experiment_root) / entry["experiment"]
    train_df = read_split_csv(experiment_dir / "train.csv", "train", seed + 10)
    val_df = read_split_csv(experiment_dir / "val.csv", "val", seed + 20)
    test_df = read_split_csv(experiment_dir / "test.csv", "test", seed + 30)

    # Exact row counts are enforced for full runs. Debug mode intentionally caps them.
    if CFG.debug_max_rows_per_split is None:
        actual = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}
        expected = {"train": 2800, "val": 560, "test": 560}
        if actual != expected:
            raise ValueError(
                f"{entry['experiment']}: CSV row counts {actual} do not match plan counts {expected}."
            )

    # Hashes—not path strings—guard against leakage and duplicate content.
    for split_name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
        if frame.sha256.duplicated().any():
            raise ValueError(f"{entry['experiment']}: duplicate content inside {split_name}.csv")
    if set(train_df.sha256) & set(val_df.sha256):
        raise ValueError(f"{entry['experiment']}: train/validation content leakage detected.")
    if set(train_df.sha256) & set(test_df.sha256):
        raise ValueError(f"{entry['experiment']}: train/test content leakage detected.")
    if set(val_df.sha256) & set(test_df.sha256):
        raise ValueError(f"{entry['experiment']}: validation/test content leakage detected.")

    train_ai_generators = set(train_df.loc[train_df.label == 1, "generator"])
    val_ai_generators = set(val_df.loc[val_df.label == 1, "generator"])
    expected_train_generators = set(entry["train_generators"])
    if train_ai_generators != expected_train_generators:
        raise ValueError(
            f"{entry['experiment']}: train AI generators {sorted(train_ai_generators)} "
            f"do not match plan {sorted(expected_train_generators)}."
        )
    if val_ai_generators != expected_train_generators:
        raise ValueError(
            f"{entry['experiment']}: validation AI generators {sorted(val_ai_generators)} "
            f"do not match plan {sorted(expected_train_generators)}."
        )
    test_ai_generators = set(test_df.loc[test_df.label == 1, "generator"])
    if test_ai_generators != {entry["holdout"]}:
        raise ValueError(
            f"{entry['experiment']}: test AI generators {sorted(test_ai_generators)} "
            f"do not match held-out generator {entry['holdout']}."
        )

    expected_per_class = {"train": 1400, "val": 280, "test": 280}
    for split_name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
        counts = frame.label.value_counts().to_dict()
        expected_counts = {0: expected_per_class[split_name], 1: expected_per_class[split_name]}
        if CFG.debug_max_rows_per_split is None and counts != expected_counts:
            raise ValueError(f"{entry['experiment']}: {split_name} class counts {counts} != {expected_counts}.")
        unknown_real = set(frame.loc[frame.label == 0, "generator"]) - set(available_generators)
        if unknown_real:
            raise ValueError(f"{entry['experiment']}: unknown real-image sources {sorted(unknown_real)}.")

    # Preserve and report source_split, but keep the dataset team's CSV membership
    # authoritative. Rows are never silently moved between experiment splits.
    return train_df, val_df, test_df, seed

## 5. Images, transformations, and loaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
AUGMENTATION = {
    "blur_p": 0.3, "blur_radius": [0.1, 2.0],
    "jpeg_p": 0.3, "jpeg_quality": [40, 95],
    "resize_p": 0.3, "resize_scale": [0.5, 1.0],
    "noise_p": 0.3, "noise_sigma": [0.005, 0.03],
    "lowpass_p": 0.3, "lowpass_cutoff": [0.35, 0.9],
}


def lowpass_image(image: Image.Image, cutoff: float) -> Image.Image:
    array = np.asarray(image, dtype=np.float32) / 255.0
    height, width = array.shape[:2]
    fy = np.fft.fftfreq(height)[:, None]
    fx = np.fft.fftfreq(width)[None, :]
    mask = (np.sqrt(fx * fx + fy * fy) <= 0.5 * cutoff)[..., None]
    filtered = np.fft.ifft2(np.fft.fft2(array, axes=(0, 1)) * mask, axes=(0, 1)).real
    return Image.fromarray(np.round(np.clip(filtered, 0, 1) * 255).astype(np.uint8))


def corrupt_image(image: Image.Image, operation: str, value: float) -> Image.Image:
    if operation == "blur":
        return image.filter(ImageFilter.GaussianBlur(radius=value))
    if operation == "jpeg":
        with io.BytesIO() as buffer:
            image.save(buffer, format="JPEG", quality=int(value), subsampling=2)
            buffer.seek(0)
            with Image.open(buffer) as decoded:
                return decoded.convert("RGB").copy()
    if operation == "resize":
        width, height = image.size
        small = image.resize((max(1, round(width * value)), max(1, round(height * value))), Image.Resampling.BICUBIC)
        return small.resize(image.size, Image.Resampling.BICUBIC)
    if operation == "noise":
        array = np.asarray(image, dtype=np.float32) / 255.0
        noisy = np.clip(array + np.random.normal(0, value, array.shape), 0, 1)
        return Image.fromarray(np.round(noisy * 255).astype(np.uint8))
    if operation == "lowpass":
        return lowpass_image(image, value)
    raise ValueError(operation)


class BenchmarkTransform:
    def __init__(self, augmented: bool = False):
        self.augmented = augmented
        self.geometry = transforms.Compose([
            transforms.Resize(CFG.image_size, interpolation=InterpolationMode.BICUBIC),
            transforms.CenterCrop(CFG.image_size),
        ])
        self.finish = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __call__(self, image: Image.Image) -> torch.Tensor:
        image = self.geometry(image)
        if self.augmented:
            specs = [("blur", "blur_radius"), ("jpeg", "jpeg_quality"),
                     ("resize", "resize_scale"), ("noise", "noise_sigma"),
                     ("lowpass", "lowpass_cutoff")]
            for operation, key in specs:
                if random.random() < AUGMENTATION[operation + "_p"]:
                    low, high = AUGMENTATION[key]
                    value = random.randint(int(low), int(high)) if operation == "jpeg" else random.uniform(low, high)
                    image = corrupt_image(image, operation, value)
        return self.finish(image)


eval_transform = BenchmarkTransform(augmented=False)
TRAIN_TRANSFORMS = {"baseline": eval_transform, "augmented": BenchmarkTransform(augmented=True)}
for variant in TRAINING_VARIANTS:
    description = "none" if variant == "baseline" else "ViT-matched blur/JPEG/resize/noise/low-pass"
    print(f"{variant} training augmentation: {description}")


class ManifestImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, data_root: str, transform):
        self.frame = frame.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        raw_path = Path(str(row.path))
        path = raw_path if raw_path.is_absolute() else self.data_root / raw_path
        try:
            with Image.open(path) as image:
                image = image.convert("RGB")
        except Exception as exc:
            raise RuntimeError(f"Unable to load {path}") from exc
        return {
            "image": self.transform(image),
            "label": torch.tensor(float(row.label), dtype=torch.float32),
            "path": str(row.path),
            "generator": str(row.generator),
            "sha256": str(row.sha256),
        }


def make_loader(frame, transform, seed, shuffle=False):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        ManifestImageDataset(frame, CFG.data_root, transform),
        batch_size=CFG.batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=CFG.num_workers > 0,
        worker_init_fn=worker_init_fn,
        generator=generator,
    )

## 6. Model, metrics, training, and prediction functions

In [ ]:
def build_model() -> nn.Module:
    weights = ResNet50_Weights.IMAGENET1K_V2 if CFG.pretrained else None
    model = resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model


def safe_auroc(labels, probabilities) -> float:
    return float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) == 2 else float("nan")


def reliability_bins(labels, probabilities, bins: int) -> pd.DataFrame:
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    indices = np.minimum((probabilities * bins).astype(int), bins - 1)
    rows = []
    for bin_index in range(bins):
        selected = indices == bin_index
        count = int(selected.sum())
        rows.append({
            "bin": bin_index, "lower": bin_index / bins, "upper": (bin_index + 1) / bins,
            "count": count,
            "mean_prob_fake": float(probabilities[selected].mean()) if count else None,
            "observed_fake_fraction": float(labels[selected].mean()) if count else None,
        })
    return pd.DataFrame(rows)


def binary_metrics(labels, probabilities, threshold=0.5) -> dict:
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if (labels.ndim != 1 or probabilities.shape != labels.shape or len(labels) == 0
            or not np.isin(labels, [0, 1]).all() or not np.isfinite(probabilities).all()
            or ((probabilities < 0) | (probabilities > 1)).any()):
        raise ValueError("Expected finite binary labels and aligned probabilities in [0,1].")
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    calibration = reliability_bins(labels, probabilities, CFG.calibration_bins)
    occupied = calibration[calibration["count"] > 0]
    ece = ((occupied.mean_prob_fake - occupied.observed_fake_fraction).abs()
           * occupied["count"]).sum() / len(labels)
    clipped = np.clip(probabilities, 1e-7, 1 - 1e-7)
    f1 = float(f1_score(labels, predictions, zero_division=0))
    return {
        "n": int(len(labels)),
        "accuracy": float(accuracy_score(labels, predictions)),
        "threshold": float(threshold),
        "precision_fake": float(precision_score(labels, predictions, zero_division=0)),
        "recall_fake": float(recall_score(labels, predictions, zero_division=0)),
        "f1_fake": f1, "f1": f1,
        "auroc": safe_auroc(labels, probabilities),
        "ece": float(ece),
        "brier": float(np.mean((probabilities - labels) ** 2)),
        "log_loss": float(-np.mean(labels * np.log(clipped) + (1-labels) * np.log1p(-clipped))),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, labels, probabilities = 0.0, [], []
    for batch in tqdm(loader, leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["label"].to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), autocast(enabled=CFG.use_amp and DEVICE.type == "cuda"):
            logits = model(images).squeeze(1)
            loss = criterion(logits, targets)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * len(targets)
        labels.extend(targets.detach().cpu().numpy())
        probabilities.extend(torch.sigmoid(logits).detach().cpu().numpy())
    metrics = binary_metrics(labels, probabilities, CFG.decision_threshold)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def predict(model, loader) -> pd.DataFrame:
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            with autocast(enabled=CFG.use_amp and DEVICE.type == "cuda"):
                probs = torch.sigmoid(model(images).squeeze(1)).cpu().numpy()
            for path, image_hash, generator, label, prob in zip(
                batch["path"], batch["sha256"], batch["generator"], batch["label"].numpy(), probs
            ):
                rows.append({
                    "path": path, "sha256": image_hash, "image_id": image_hash,
                    "generator": generator, "label": int(label),
                    "probability_fake": float(prob),
                })
    result = pd.DataFrame(rows)
    result["prediction"] = (result.probability_fake >= CFG.decision_threshold).astype(int)
    return result

## 7. Execute one experiment

In [ ]:
def run_experiment(entry: dict, variant: str) -> dict:
    if variant not in TRAIN_TRANSFORMS:
        raise ValueError(f"Unknown training variant: {variant}")
    seed = CFG.base_seed
    base_experiment = entry["experiment"]
    run_name = f"{base_experiment}__{variant}__seed{seed}"
    run_group = "debug_runs" if CFG.debug_max_rows_per_split is not None else "runs"
    run_dir = Path(CFG.output_root) / run_group / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    summary_path = run_dir / "summary.json"
    complete_path = run_dir / "TRAINING_COMPLETE.json"
    train_df, val_df, test_df, resolved_seed = load_experiment_splits(entry)
    if resolved_seed != seed:
        raise RuntimeError("Resolved seed differs from the configured benchmark seed.")
    seed_everything(seed)
    augmentation = None if variant == "baseline" else AUGMENTATION
    protocol_config = {key: value for key, value in asdict(CFG).items()
                       if key not in {"output_root", "skip_completed", "max_runs_this_session"}}
    protocol = {
        **entry, "run_name": run_name, "mode": variant, "seed": seed,
        "model": "torchvision/resnet50/IMAGENET1K_V2",
        "fully_finetuned": True, "augmentation": augmentation,
        "label_mapping": {"real": 0, "fake": 1},
        "selection": "validation AUROC; ties within tolerance use validation BCE",
        "config": protocol_config,
    }
    protocol_hash = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()
    protocol["protocol_hash"] = protocol_hash
    config_path = run_dir / "config.json"
    if config_path.exists():
        previous = json.loads(config_path.read_text())
        if previous.get("protocol_hash") != protocol_hash:
            raise ValueError(f"Protocol changed for {run_dir}; use a new output root.")
    else:
        config_path.write_text(json.dumps(protocol, indent=2))
    if CFG.skip_completed and summary_path.is_file() and complete_path.is_file():
        marker = json.loads(complete_path.read_text())
        if marker.get("protocol_hash") != protocol_hash or not (run_dir / "best.pt").is_file():
            raise ValueError(f"Invalid completed-run marker: {run_dir}")
        print(f"Verified completed run; skipping {run_name}")
        return json.loads(summary_path.read_text())

    print("\n" + "=" * 88)
    print(run_name)
    print("Mode:", variant, "| holdout:", entry["holdout"], "| k:", entry["k"], "| rep:", entry["rep"])
    print("Train generators:", entry["train_generators"], "| seed:", seed)
    print("Exact split sizes:", {"train": len(train_df), "val": len(val_df), "test": len(test_df)})
    train_df.to_csv(run_dir / "input_train.csv", index=False)
    val_df.to_csv(run_dir / "input_val.csv", index=False)
    test_df.to_csv(run_dir / "input_test.csv", index=False)

    model = build_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scaler = GradScaler(enabled=CFG.use_amp and DEVICE.type == "cuda")
    history, best_auroc, best_val_loss, no_improvement, start_epoch = [], -np.inf, np.inf, 0, 1
    last_path = run_dir / "last.pt"
    if last_path.exists():
        saved = torch.load(last_path, map_location=DEVICE)
        if saved.get("protocol_hash") != protocol_hash:
            raise ValueError(f"Checkpoint protocol mismatch: {last_path}")
        model.load_state_dict(saved["model_state_dict"])
        optimizer.load_state_dict(saved["optimizer_state_dict"])
        scaler.load_state_dict(saved["scaler_state_dict"])
        history = saved["history"]
        best_auroc, best_val_loss = saved["best_val_auroc"], saved["best_val_loss"]
        no_improvement, start_epoch = saved["no_improvement"], saved["epoch"] + 1
        print(f"Resuming after epoch {start_epoch - 1}")

    val_loader = make_loader(val_df, eval_transform, seed, shuffle=False)
    for epoch in range(start_epoch, CFG.epochs + 1):
        if no_improvement >= CFG.early_stopping_patience:
            break
        seed_everything(seed + epoch * 1009)
        train_loader = make_loader(train_df, TRAIN_TRANSFORMS[variant], seed + epoch, shuffle=True)
        started = time.time()
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_metrics = run_epoch(model, val_loader, criterion)
        row = {"epoch": epoch, "learning_rate": optimizer.param_groups[0]["lr"],
               "seconds": time.time() - started,
               **{f"train_{key}": value for key, value in train_metrics.items()},
               **{f"val_{key}": value for key, value in val_metrics.items()}}
        history.append(row)
        score, val_loss = val_metrics["auroc"], val_metrics["log_loss"]
        improved = (np.isfinite(score) and (score > best_auroc + CFG.auroc_tolerance or
                    (abs(score - best_auroc) <= CFG.auroc_tolerance and val_loss < best_val_loss)))
        if improved:
            best_auroc, best_val_loss, no_improvement = score, val_loss, 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "best_val_auroc": best_auroc, "best_val_loss": best_val_loss,
                        "protocol_hash": protocol_hash}, run_dir / "best.pt")
        else:
            no_improvement += 1
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(), "history": history,
                    "best_val_auroc": best_auroc, "best_val_loss": best_val_loss,
                    "no_improvement": no_improvement, "protocol_hash": protocol_hash}, last_path)
        pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)
        print(f"Epoch {epoch:02d} | train loss {train_metrics['loss']:.4f} | "
              f"val loss {val_metrics['loss']:.4f} | val F1 {val_metrics['f1_fake']:.4f} | "
              f"val AUROC {val_metrics['auroc']:.4f}")
        del train_loader

    best = torch.load(run_dir / "best.pt", map_location=DEVICE)
    model.load_state_dict(best["model_state_dict"])
    validation_predictions = predict(model, val_loader)
    validation_predictions.to_csv(run_dir / "val_predictions.csv", index=False)
    selected_val_metrics = binary_metrics(
        validation_predictions.label, validation_predictions.probability_fake, CFG.decision_threshold)
    (run_dir / "val_metrics.json").write_text(json.dumps(selected_val_metrics, indent=2))
    reliability_bins(validation_predictions.label, validation_predictions.probability_fake,
                     CFG.calibration_bins).to_csv(run_dir / "val_calibration.csv", index=False)
    test_loader = make_loader(test_df, eval_transform, seed, shuffle=False)
    predictions = predict(model, test_loader)
    for key, value in {"experiment": base_experiment, "holdout": entry["holdout"],
                       "k": int(entry["k"]), "rep": int(entry["rep"]),
                       "mode": variant, "seed": seed}.items():
        predictions[key] = value
    predictions["threshold"] = CFG.decision_threshold
    predictions.to_csv(run_dir / "test_predictions.csv", index=False)
    test_metrics = binary_metrics(predictions.label, predictions.probability_fake, CFG.decision_threshold)
    (run_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
    reliability_bins(predictions.label, predictions.probability_fake, CFG.calibration_bins).to_csv(
        run_dir / "test_calibration.csv", index=False)
    result = {"run_name": run_name, "experiment": base_experiment, "mode": variant,
              "variant": variant, "holdout": entry["holdout"], "test_generator": entry["holdout"],
              "k": int(entry["k"]), "rep": int(entry["rep"]), "seed": seed,
              "train_generators": "+".join(entry["train_generators"]),
              "best_epoch": int(best["epoch"]), "best_val_auroc": float(best["best_val_auroc"]),
              "protocol_hash": protocol_hash,
              **{f"test_{key}": value for key, value in test_metrics.items()}}

    history_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history_df.epoch, history_df.train_loss, label="train")
    axes[0].plot(history_df.epoch, history_df.val_loss, label="validation")
    axes[0].set(title="Loss", xlabel="Epoch")
    axes[1].plot(history_df.epoch, history_df.train_auroc, label="train")
    axes[1].plot(history_df.epoch, history_df.val_auroc, label="validation")
    axes[1].set(title="AUROC", xlabel="Epoch", ylim=(0, 1))
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    fig.suptitle(run_name)
    fig.tight_layout()
    fig.savefig(run_dir / "training_curves.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    summary_path.write_text(json.dumps(result, indent=2))
    complete_path.write_text(json.dumps({"protocol_hash": protocol_hash, "best_epoch": int(best["epoch"])}, indent=2))
    last_path.unlink(missing_ok=True)  # Resume state is unnecessary after successful completion.

    del model, optimizer, scaler, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 8. Run the selected list

In [ ]:
selected_runs = [(experiment, variant) for experiment in selected_experiments for variant in TRAINING_VARIANTS]
if CFG.max_runs_this_session is not None:
    selected_runs = selected_runs[:CFG.max_runs_this_session]
print(f"Selected {len(selected_runs)} CNN training runs.")
completed_results = []
for experiment, variant in selected_runs:
    try:
        completed_results.append(run_experiment(experiment, variant))
    except Exception as exc:
        print(f"FAILED: {experiment['experiment']} [{variant}]: {exc}")
        raise

results_df = pd.DataFrame(completed_results)
Path(CFG.output_root).mkdir(parents=True, exist_ok=True)
results_df.to_csv(Path(CFG.output_root) / "selected_experiment_results.csv", index=False)
display(results_df)

## 9. Consolidate every completed run

In [ ]:
summary_files = sorted((Path(CFG.output_root) / "runs").glob("*/summary.json"))
if summary_files:
    all_results = pd.DataFrame([json.loads(path.read_text()) for path in summary_files])
    all_results.to_csv(Path(CFG.output_root) / "all_experiment_results.csv", index=False)
    print(f"Completed benchmark runs: {len(all_results)}/192")
    if len(all_results) != 192:
        print("WARNING: summaries are partial until all 192 runs are complete.")
    display(all_results.sort_values(["holdout", "k", "variant", "rep"]))

    aggregate = (
        all_results.groupby(["holdout", "k", "variant"], as_index=False)
        .agg(
            mean_test_accuracy=("test_accuracy", "mean"),
            std_test_accuracy=("test_accuracy", "std"),
            mean_test_f1=("test_f1", "mean"),
            std_test_f1=("test_f1", "std"),
            mean_test_auroc=("test_auroc", "mean"),
            std_test_auroc=("test_auroc", "std"),
            n_runs=("run_name", "count"),
        )
    )
    aggregate.to_csv(Path(CFG.output_root) / "aggregate_by_holdout_k_and_variant.csv", index=False)
    equal_weight_holdouts = (
        aggregate.groupby(["k", "variant"], as_index=False)
        .agg(mean_holdout_accuracy=("mean_test_accuracy", "mean"),
             mean_holdout_f1=("mean_test_f1", "mean"),
             mean_holdout_auroc=("mean_test_auroc", "mean"),
             n_holdouts=("holdout", "nunique"))
    )
    equal_weight_holdouts.to_csv(Path(CFG.output_root) / "equal_weight_holdout_summary.csv", index=False)
    display(aggregate)
    display(equal_weight_holdouts)
else:
    print("No completed summaries found.")

## How to choose runs

One manifest experiment (two runs when both modes are enabled):

```python
SELECTED_EXPERIMENTS = ["holdout_ADM_k1_rep0"]
```

Two runs:

```python
SELECTED_EXPERIMENTS = [
    "holdout_ADM_k1_rep0",
    "holdout_ADM_k2_rep0",
]
```

Every corrected manifest (96 experiments × 2 modes = 192 runs):

```python
SELECTED_EXPERIMENTS = ["ALL"]
```

To process the full plan in resumable chunks, set `CFG.max_runs_this_session` to a positive number. Completed runs are skipped on the next execution. For a quick integration check, first set:

```python
CFG.debug_max_rows_per_split = 200
CFG.epochs = 1
CFG.batch_size = 32
```

Then restore the full training settings. For all reportable runs, keep:

```python
CFG.debug_max_rows_per_split = None
```

This ensures the supplied CSV rows are used exactly. Debug mode is only for a short integration check.

## ViT/CNN alignment used here

- Exact same experiment CSV rows
- `0=real`, `1=fake`
- Bicubic resize to 224 and center crop to 224
- No random augmentation for the primary baseline
- The same ViT blur/JPEG/resize/noise/low-pass recipe for augmented training
- One matched classifier/training seed: 17; repeats represent manifest variation
- Model-specific normalization: ImageNet for ResNet, CLIP statistics for ViT
- ResNet is fully fine-tuned from ImageNet-pretrained weights; CLIP remains frozen


## Choose baseline, augmentation, or both

Baseline only:

```python
TRAINING_VARIANTS = ["baseline"]
```

Augmented only:

```python
TRAINING_VARIANTS = ["augmented"]
```

Run both independently on every selected experiment:

```python
TRAINING_VARIANTS = ["baseline", "augmented"]
```

Outputs are isolated in directories such as:

```text
/project2/yzhao010_1531/csci566-genimage/results/cnn_benchmark/notebook/runs/holdout_ADM_k1_rep0__baseline__seed17/
/project2/yzhao010_1531/csci566-genimage/results/cnn_benchmark/notebook/runs/holdout_ADM_k1_rep0__augmented__seed17/
```

Both variants use the same CSV rows, seed, optimizer, learning rate, epoch limit, and evaluation transform. Only the training-image augmentation changes. Completed runs are skipped and interrupted runs resume after their last saved epoch.
